# Optimize tables

- OPTIMIZE on all Delta tables
- VACUUM on Silver and Gold
- ANALYZE for statistics

Setup and Configuration

In [0]:
from pyspark.sql import SparkSession
from datetime import datetime, timezone

spark = SparkSession.builder.getOrCreate()
print(f"Optimization started: {datetime.now(timezone.utc).isoformat()}")

Define all tables

In [0]:
# Bronze table
BRONZE_RAW_DELIVERY_TABLE = "ipl_catalog.bronze.raw_deliveries"
BRONZE_RAW_MATCH_INFO_TABLE = "ipl_catalog.bronze.raw_match_info"
BRONZE_RAW_PLAYER_TABLE = "ipl_catalog.bronze.raw_players"

# Silver tables
SILVER_DELIVERIES_TABLE = "ipl_catalog.silver.deliveries"
SILVER_MATCHES_TABLE = "ipl_catalog.silver.matches"
SILVER_PLAYERS_TABLE = "ipl_catalog.silver.players"
SILVER_VENUES_TABLE = "ipl_catalog.silver.venues"

# Gold tables
GOLD_PLAYER_SEASONS_TABLE = "ipl_catalog.gold.player_season_stats"
GOLD_TEAM_WIN_RATES_TABLE = "ipl_catalog.gold.team_win_rates"
GOLD_VENUE_AVG_SCORES_TABLE = "ipl_catalog.gold.venue_avg_scores"
GOLD_HEAD_TO_HEAD_TABLE = "ipl_catalog.gold.head_to_head"
GOLD_BOWLER_ECONOMY_TABLE = "ipl_catalog.gold.bowler_economy"
GOLD_MATCH_SUMMARY_TABLE = "ipl_catalog.gold.match_summary"

BRONZE_TABLES = [
    BRONZE_RAW_DELIVERY_TABLE,
    BRONZE_RAW_MATCH_INFO_TABLE,
    BRONZE_RAW_PLAYER_TABLE,
]

SILVER_TABLES = [
    SILVER_DELIVERIES_TABLE,
    SILVER_MATCHES_TABLE,
    SILVER_PLAYERS_TABLE,
    SILVER_VENUES_TABLE,
]

GOLD_TABLES = [
    GOLD_PLAYER_SEASONS_TABLE,
    GOLD_TEAM_WIN_RATES_TABLE,
    GOLD_VENUE_AVG_SCORES_TABLE,
    GOLD_HEAD_TO_HEAD_TABLE,
    GOLD_BOWLER_ECONOMY_TABLE,
    GOLD_MATCH_SUMMARY_TABLE,
]

ALL_TABLES = BRONZE_TABLES + SILVER_TABLES + GOLD_TABLES

OPTIMIZE all tables
- OPTIMIZE compacts small files into larger ones, makes queries significantly faster, should run after every batch load

In [0]:
print("Running OPTIMIZE on all tables...")
print("=" * 50)

optimize_results = {}

for table in ALL_TABLES:
    try:
        spark.sql(f"OPTIMIZE {table}")
        optimize_results[table] = "✓ optimized"
        print(f"  ✓ {table}")
    except Exception as e:
        optimize_results[table] = f"✗ failed: {e}"
        print(f"  ✗ {table} — {e}")

print("\nOPTIMIZE complete")

ZORDER key columns
- ZORDER co-locates related data in same files
- makes filtering by these columns much faster

In [0]:
print("Running ZORDER on large tables...")
print("=" * 50)

zorder_configs = {
    BRONZE_RAW_DELIVERY_TABLE : "match_id, season",
    SILVER_DELIVERIES_TABLE : "match_id, season, batting_team",
    SILVER_MATCHES_TABLE : "match_id, season",
    SILVER_PLAYERS_TABLE : "player_id, season",
    GOLD_PLAYER_SEASONS_TABLE : "player_name, season",
    GOLD_TEAM_WIN_RATES_TABLE : "team, season",
    GOLD_BOWLER_ECONOMY_TABLE : "player_name, season, phase",
}

for table, zorder_cols in zorder_configs.items():
    try:
        spark.sql(f"OPTIMIZE {table} ZORDER BY ({zorder_cols})")
        print(f"  ✓ {table} ZORDER BY ({zorder_cols})")
    except Exception as e:
        print(f"  ✗ {table} — {e}")

print("\nZORDER complete")

VACUUM old versions
- VACUUM removes old Delta lake versions
- keeps last 7 days by default (168 hours)
- Frees up storage space

In [0]:
# print("Running VACUUM on Silver and Gold tables...")
# print("=" * 50)

# vacuum_tables = SILVER_TABLES + GOLD_TABLES

# for table in vacuum_tables:
#     try:
#         spark.sql(f"VACUUM {table} RETAIN 168 HOURS")
#         print(f"  ✓ {table}")
#     except Exception as e:
#         print(f"  ✗ {table} — {e}")

# print("\nVACUUM complete")

ANALYZE for query statistics
- ANALYZE computes statistics that Spark uses for query planning
- Makes joins and aggregations faster

In [0]:
print("Running ANALYZE on Gold tables...")
print("=" * 50)

for table in GOLD_TABLES:
    try:
        spark.sql(
            f"ANALYZE TABLE {table} COMPUTE STATISTICS FOR ALL COLUMNS"
        )
        print(f"  ✓ {table}")
    except Exception as e:
        print(f"  ✗ {table} — {e}")

print("\nANALYZE complete")

Print table sizes

In [0]:
print("\nTable statistics after optimization:")
print("=" * 50)

for table in ALL_TABLES:
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {table}").collect()[0]
        size_mb = detail["sizeInBytes"] / (1024 * 1024)
        num_files = detail["numFiles"]
        print(f"  {table.split('.')[-1]:30} "
              f"size={size_mb:.1f}MB  files={num_files}")
    except Exception as e:
        print(f"  {table.split('.')[-1]:30} ERROR: {e}")

print(f"\n04_optimize_tables complete: {datetime.utcnow().isoformat()}")